
# Feature Engineering for Startup Funding Analysis

## 1. Introduction

Feature engineering is the process of creating new meaningful features from existing data to improve data analysis and machine learning model performance.

In this notebook, we will create useful features from the cleaned startup funding dataset to support further analysis and predictive modeling.

### Objectives

- Extract year and month from funding dates.
- Transform funding amounts to handle their skewed distribution.
- Group less frequent categories where appropriate.
- Prepare relevant features for machine learning models.



## 2. Import Required Libraries

We import the required Python libraries for data manipulation, numerical operations and visualization.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")


Libraries imported successfully!



## 3. Load the Cleaned Dataset

We load the preprocessed startup funding dataset saved during the data cleaning stage.

This dataset contains the original columns along with the cleaned columns that will be used for feature engineering.


In [2]:

file_path = "../data/processed/startup_funding_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset shape: (3044, 21)


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks,...,Date_Cleaned,Amount_Approximate,Amount_Cleaned,City_Cleaned,Industry_Cleaned,SubVertical_Cleaned,Investment_Type_Cleaned,Investors_Cleaned,Remarks_Cleaned,Is_Outlier
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN,...,2020-01-09,False,200000000.0,Bengaluru,E-Tech,E-Learning,Private Equity,Tiger Global Management,NaN,True
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN,...,2020-01-13,False,8048394.0,Gurugram,Transportation,App Based Shuttle Service,Series C,Susquehanna Growth Equity,NaN,False
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN,...,2020-01-09,False,18358860.0,Bengaluru,Ecommerce,Retailer Of Baby And Toddler Products,Series B,Sequoia Capital India,NaN,False
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN,...,2020-01-02,False,3000000.0,New Delhi,Fintech,Online Investment,Pre-Series A,Vinod Khatumal,NaN,False
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN,...,2020-01-02,False,1800000.0,Mumbai,Fashion And Apparel,Embroiled Clothes For Women,Seed Round,Sprout Venture Partners,NaN,False



## 4. Extract Funding Month

The month of a funding record can help us identify monthly patterns in startup funding activity.

We extract the month from the cleaned funding date and create a new feature called `Month`.


In [3]:


df["Date_Cleaned"] = pd.to_datetime(
    df["Date_Cleaned"],
    errors="coerce"
)

print(df["Date_Cleaned"].dtype)



datetime64[ns]


In [4]:

df["Month"] = df["Date_Cleaned"].dt.month

display(df[["Date_Cleaned", "Month"]].head())


,Date_Cleaned,Month
0,2020-01-09,1
1,2020-01-13,1
2,2020-01-09,1
3,2020-01-02,1
4,2020-01-02,1



## 5. Analysis of Categorical Features

Categorical features contain descriptive information such as city, industry and investment type.

We examine their unique values and frequencies to understand the categories before preparing them for machine learning.


In [5]:

categorical_columns = [
    "City_Cleaned",
    "Industry_Cleaned",
    "Investment_Type_Cleaned"
]

for col in categorical_columns:
    print("\n", col)
    print("Number of unique categories:", df[col].nunique())
    display(df[col].value_counts().head(10))



 City_Cleaned
Number of unique categories: 101


City_Cleaned
Bengaluru    842
Mumbai       568
New Delhi    425
Gurugram     341
Unknown      180
Pune         105
Hyderabad     99
Chennai       97
Noida         93
Ahmedabad     41
Name: count, dtype: int64


 Industry_Cleaned
Number of unique categories: 793


Industry_Cleaned
Consumer Internet    942
Technology           478
Ecommerce            299
Unknown              171
Healthcare            71
Finance               62
Food & Beverage       35
Logistics             32
Education             24
EdTech                20
Name: count, dtype: int64


 Investment_Type_Cleaned
Number of unique categories: 40


Investment_Type_Cleaned
Seed Funding          1387
Private Equity        1362
Seed/Angel Funding     141
Debt Funding            26
Series A                24
Series B                20
Series C                14
Series D                12
Pre-Series A             9
Seed Round               7
Name: count, dtype: int64


## 6. Grouping Rare Categories

Some categorical features contain many categories with very few records.

To reduce unnecessary categories and simplify future machine learning encoding, we group categories with fewer than 20 records into a new category called "Other".

We keep "Unknown" separate because it represents missing information.


In [6]:

city_counts = df["City_Cleaned"].value_counts()

rare_cities = city_counts[
    (city_counts < 20) & (city_counts.index != "Unknown")
].index

df["City_Grouped"] = df["City_Cleaned"].replace(
    rare_cities, "Other"
)

print("Original number of city categories:",
      df["City_Cleaned"].nunique())

print("Number of city categories after grouping:",
      df["City_Grouped"].nunique())

display(df["City_Grouped"].value_counts())


Original number of city categories: 101
Number of city categories after grouping: 14


City_Grouped
Bengaluru    842
Mumbai       568
New Delhi    425
Gurugram     341
Unknown      180
Other        167
Pune         105
Hyderabad     99
Chennai       97
Noida         93
Ahmedabad     41
Delhi         34
Jaipur        30
Kolkata       22
Name: count, dtype: int64


## 7. Grouping Rare Industry Categories

The industry feature contains many unique categories. To reduce high dimensionality during machine learning encoding, we group industries with fewer than 20 records into "Other".

The "Unknown" category is preserved separately.


In [12]:

# Keep all original cleaned industry names
df["Industry_Grouped"] = df["Industry_Cleaned"]

print("Original number of industry categories:",
      df["Industry_Cleaned"].nunique())

print("Number of industry categories after grouping:",
      df["Industry_Grouped"].nunique())

display(df["Industry_Grouped"].value_counts())

Original number of industry categories: 793
Number of industry categories after grouping: 793


Industry_Grouped
Consumer Internet                 942
Technology                        478
Ecommerce                         299
Unknown                           171
Healthcare                         71
                                 ... 
Mobile Food Ordering App            1
Financial Markets Software          1
Hiring Analytics Platform           1
Online Travel Marketplace           1
Restaurant Management Platform      1
Name: count, Length: 793, dtype: int64

In [14]:
# Check funding records available for each industry

industry_funding_counts = (
    df[df["Industry_Grouped"] != "Unknown"]
    .dropna(subset=["Amount_Cleaned"])
    .groupby("Industry_Grouped")["Amount_Cleaned"]
    .count()
    .sort_values(ascending=False)
)

display(industry_funding_counts.head(15))

Industry_Grouped
Consumer Internet               590
Technology                      310
Ecommerce                       205
Finance                          57
Healthcare                       45
Food & Beverage                  25
Logistics                        23
Education                        19
EdTech                           18
Fintech                          16
IT                               11
Health And Wellness               6
Online Food Delivery              5
Logistics Tech                    4
Hyperlocal Handyman Services      4
Name: Amount_Cleaned, dtype: int64

In [15]:
from scipy.stats import kruskal

# Select industries with at least 15 valid funding records
selected_industries = industry_funding_counts[
    industry_funding_counts >= 15
].index

# Prepare funding amount groups
funding_groups = [
    df[
        (df["Industry_Grouped"] == industry) &
        (df["Amount_Cleaned"].notna())
    ]["Amount_Cleaned"].values
    for industry in selected_industries
]

# Perform Kruskal-Wallis test
kruskal_result = kruskal(*funding_groups)

print("Kruskal-Wallis Test")
print("Number of industries:", len(selected_industries))
print("Industries:", list(selected_industries))
print("H-statistic:", kruskal_result.statistic)
print("P-value:", kruskal_result.pvalue)

Kruskal-Wallis Test
Number of industries: 10
Industries: ['Consumer Internet', 'Technology', 'Ecommerce', 'Finance', 'Healthcare', 'Food & Beverage', 'Logistics', 'Education', 'EdTech', 'Fintech']
H-statistic: 64.7611330822503
P-value: 1.6055946967317464e-10


### Kruskal-Wallis Statistical Hypothesis Test

**Objective:** To compare funding amount distributions across the 10 most frequent known industries.

**Hypotheses:**
- H0: Funding amount distributions are the same across the selected industries.
- H1: At least one industry's funding amount distribution differs.

**Test Results:**
- Number of industries: 10
- H-statistic: 64.7611
- P-value: 1.6056e-10
- Significance level: 0.05

**Interpretation:**

Since the p-value is less than 0.05, we reject the null hypothesis. There is statistically significant evidence that funding amount distributions differ across the selected industries.

**Conclusion:**

Funding amount distributions are not the same across all selected industries. However, this test does not identify exactly which industries differ.


## 8. Grouping Rare Investment Types

The investment type feature contains several categories with very few records.

We group categories with fewer than 20 records into "Other" to simplify the feature and make it easier to encode for machine learning.

The "Unknown" category is preserved separately.


In [8]:

investment_counts = df["Investment_Type_Cleaned"].value_counts()

rare_investments = investment_counts[
    (investment_counts < 20) &
    (investment_counts.index != "Unknown")
].index

df["Investment_Type_Grouped"] = df[
    "Investment_Type_Cleaned"
].replace(rare_investments, "Other")

print("Original number of investment categories:",
      df["Investment_Type_Cleaned"].nunique())

print("Number of investment categories after grouping:",
      df["Investment_Type_Grouped"].nunique())

display(df["Investment_Type_Grouped"].value_counts())


Original number of investment categories: 40
Number of investment categories after grouping: 8


Investment_Type_Grouped
Seed Funding          1387
Private Equity        1362
Seed/Angel Funding     141
Other                   80
Debt Funding            26
Series A                24
Series B                20
Unknown                  4
Name: count, dtype: int64

In [9]:

new_features = [
    "Month",
    "City_Grouped",
    "Industry_Grouped",
    "Investment_Type_Grouped"
]

print("New features created:")
display(df[new_features].head())

print("\nMissing values in new features:")
display(df[new_features].isnull().sum())


New features created:


,Month,City_Grouped,Industry_Grouped,Investment_Type_Grouped
0,1,Bengaluru,Other,Private Equity
1,1,Gurugram,Other,Other
2,1,Bengaluru,Ecommerce,Series B
3,1,New Delhi,Other,Other
4,1,Mumbai,Other,Other



Missing values in new features:


Month                      0
City_Grouped               0
Industry_Grouped           0
Investment_Type_Grouped    0
dtype: int64

In [10]:

df["Year"] = df["Date_Cleaned"].dt.year

display(df[["Date_Cleaned", "Year", "Month"]].head())

print("Year feature created successfully!")


,Date_Cleaned,Year,Month
0,2020-01-09,2020,1
1,2020-01-13,2020,1
2,2020-01-09,2020,1
3,2020-01-02,2020,1
4,2020-01-02,2020,1


Year feature created successfully!



## 9. Save the Feature-Engineered Dataset

After creating and validating the new features, we save the updated dataset separately.

This dataset will be used in the machine learning stage.


In [13]:

import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/startup_funding_feature_engineered.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Final dataset shape:", df.shape)


Feature-engineered dataset saved successfully!
Final dataset shape: (3044, 26)
